# BIA-Bilingual-Net: training notebook (Kaggle & Colab)

Train the **Darija ⇄ English** BIALinguaNet translator on a **free Kaggle or Colab GPU**.
The notebook is **self-contained**: section 2 writes the `translation_engine` package
(identical to the repo), and training goes through the same `fit()` loop as
`python -m translation_engine.train`, with the **same default configuration**.
Just upload it and press **Run all**.

| | Kaggle | Colab |
|---|---|---|
| GPU | *Settings → Accelerator → GPU T4 x2* (or P100) | *Runtime → Change runtime type → T4 GPU* |
| Internet | *Settings → Internet → On* (to download data) | on by default |
| Session limit | 12 h per session, 30 h GPU / week | ~12 h, can disconnect when idle |
| Checkpoints kept in | `/kaggle/working` (saved with "Save Version") | Google Drive if `USE_GOOGLE_DRIVE = True` |

The first run downloads the DODa corpus (~3 MB, ~45k sentence pairs) and the GPT-2 tokenizer.
One run trains **one direction** (`DIRECTION`); run it twice to get both models the app uses.

**Sections:** 1. Install · 2. The package · 3. Settings · 4. Data · 5. Model & sanity checks ·
6. Overfit a small chunk · 7. Training · 8. Results · 9. Download

## 1. Install

Kaggle and Colab already have PyTorch, transformers and matplotlib; nothing to install.

In [ ]:
import importlib.util
import sys

missing = [m for m in ("torch", "transformers", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. The package

The cells below write `translation_engine/*.py`, copied from the repo when this notebook was
built (`python notebooks/build_notebook.py`). Nothing to change here.

In [ ]:
import os

for folder in ("translation_engine/core", "translation_engine/data", "translation_engine/model"):
    os.makedirs(folder, exist_ok=True)

In [ ]:
%%writefile translation_engine/__init__.py
"""BIA-Bilingual-Net: Moroccan Darija <-> English translation with BIALinguaNet."""

In [ ]:
%%writefile translation_engine/core/__init__.py
"""Configuration (all hyper-parameters)."""

In [ ]:
%%writefile translation_engine/core/config.py
import os
import torch
from translation_engine.model.utils import get_lr


class Config:
    """
    All hyper-parameters in one place. Any attribute can be overridden with a keyword,
    e.g. Config(s_suffix="en", t_suffix="dr", n_steps=1000). Values that depend on
    others (h_size, batches_per_step, checkpoints_path, lr) are derived after the overrides.
    """

    def __init__(self, **overrides):
        # Data parameters
        self.data_dir = './translation_engine/src/dataset'
        self.s_suffix = "dr"  # source language key in the json files
        self.t_suffix = "en"  # target language key in the json files
        self.tokenizer_name = "gpt2"

        # Model parameters
        self.ev_size = 1000  # replaced by the tokenizer size (set_vocab_size)
        self.dv_size = 1000
        self.n_emb = 64
        self.n_head = 16
        self.h_size = None  # None = n_emb // n_head
        self.n_block = 1
        self.exp_fac = 3
        self.max_seq_len = 128
        self.d_rate = 0.2

        # Learning parameters
        self.checkpoints_dir = './translation_engine/src/checkpoints'
        self.checkpoints_path = None  # None = <checkpoints_dir>/<s>_<t>_chpts.pth.tar
        self.toks_in_batch = 1000  # target tokens per batch
        self.batches_per_step = None  # None = 25000 // toks_in_batch (~25k tokens per optimizer step)
        self.n_steps = 100000  # total optimizer steps (sets the number of epochs)
        self.epochs = None  # None = derived from n_steps
        self.warmup_steps = 8000
        self.step = 1
        self.lr = None  # initial lr; None = get_lr(step)
        self.start_epoch = 0
        self.betas = (0.9, 0.98)
        self.eps = 1e-9
        self.weight_decay = 1e-4
        self.grad_clip = 1.0
        self.label_smoothing = 0.1
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.cudnn_benchmark = False

        # Fine-tuning parameters
        self.finetune_lr_scale = 0.1  # fine-tune with 10x smaller learning rates
        self.finetune_epochs = 10

        for key, value in overrides.items():
            if not hasattr(self, key):
                raise AttributeError(f"Unknown config option: {key}")
            setattr(self, key, value)

        if self.h_size is None:
            self.h_size = self.n_emb // self.n_head
        if self.batches_per_step is None:
            self.batches_per_step = max(1, 25000 // self.toks_in_batch)
        if self.checkpoints_path is None:
            self.checkpoints_path = os.path.join(
                self.checkpoints_dir, f"{self.s_suffix}_{self.t_suffix}_chpts.pth.tar"
            )
        if self.lr is None:
            self.lr = get_lr(step=self.step, n_emb=self.n_emb, warmup_steps=self.warmup_steps)

    def set_vocab_size(self, tokenizer):
        # len(tokenizer) counts the added special tokens (<PAD>, <SOS>, <EOS>)
        self.ev_size = len(tokenizer)
        self.dv_size = self.ev_size

    def __repr__(self):
        return (f"<Config data_dir={self.data_dir}, direction={self.s_suffix}->{self.t_suffix}, "
                f"checkpoint_path={self.checkpoints_path}, "
                f"ev_size={self.ev_size}, dv_size={self.dv_size}, n_emb={self.n_emb}, n_head={self.n_head}, "
                f"h_size={self.h_size}, n_block={self.n_block}, exp_fac={self.exp_fac}, "
                f"max_seq_len={self.max_seq_len}, d_rate={self.d_rate}, toks_in_batch={self.toks_in_batch}, "
                f"batches_per_step={self.batches_per_step}, n_steps={self.n_steps}, "
                f"warmup_steps={self.warmup_steps}, lr={self.lr:.2e}, start_epoch={self.start_epoch}, "
                f"betas={self.betas}, eps={self.eps}, weight_decay={self.weight_decay}, "
                f"grad_clip={self.grad_clip}, label_smoothing={self.label_smoothing}, device={self.device}, "
                f"cudnn_benchmark={self.cudnn_benchmark}>")

In [ ]:
%%writefile translation_engine/data/__init__.py
"""Dataset preparation and the token-budget data loader."""

In [ ]:
%%writefile translation_engine/data/dataloader.py
import json
from transformers import AutoTokenizer
import os
import torch
from random import shuffle
from itertools import groupby
from torch.nn.utils.rnn import pad_sequence


SPECIAL_TOKENS = {"pad_token": "<PAD>", "bos_token": "<SOS>", "eos_token": "<EOS>"}


def build_tokenizer(name: str = "gpt2"):
    """
    GPT-2 BPE tokenizer + <PAD>, <SOS>, <EOS>. Both languages share this vocabulary,
    which is what lets the model tie its embeddings. Training, fine-tuning and the
    app must all use this same tokenizer.
    """
    tokenizer = AutoTokenizer.from_pretrained(name)
    tokenizer.add_special_tokens(SPECIAL_TOKENS)
    tokenizer.model_max_length = int(1e9)  # we filter long sentences ourselves
    return tokenizer


class DataLoader(object):
    """
    A custom DataLoader for loading, processing, and batching data for sequence-to-sequence tasks using the GPT-2 tokenizer.

    This DataLoader reads data from a JSON file, tokenizes the input and target sequences, and creates batches
    of data suitable for training or evaluation of sequence models. Sentences of similar lengths are grouped
    so that each batch holds about `toks_in_batch` target tokens with little padding.

    Attributes:
        data_dir (str): The directory where the dataset is located.
        s_suffix (str): The key for the source sequences in the JSON data.
        t_suffix (str): The key for the target sequences in the JSON data.
        split (str): The dataset split to load (train, val, test, etc.).
        toks_in_batch (int): The maximum number of target tokens in each batch.
        tokenizer (AutoTokenizer): The tokenizer used for tokenizing sequences.
        max_seq_len (int, optional): Pairs whose source or target is longer than this are dropped
                                     (the model cannot embed positions beyond its max_seq_len).
        limit (int, optional): Only use the first `limit` pairs of the file.
        for_training (bool): Whether the DataLoader is in training mode (batches are shuffled every epoch).
        data (list): A list of tuples (source ids, target ids, source length, target length).
        batches (list): A list of batches, each containing a subset of the data.
        n_batches (int): The number of batches created.
        current_batch (int): The index of the current batch being processed.

    Methods:
        create_batches(): Creates batches of data by grouping sequences with similar lengths and shuffling them for training.
        __iter__(): Returns an iterator over the DataLoader (restarts from the first batch).
        __next__(): Returns the next batch of data, including padded source and target sequences and their lengths.
    """

    def __init__(
        self,
        data_dir,
        s_suffix,
        t_suffix,
        split,
        toks_in_batch,
        tokenizer=None,
        max_seq_len=None,
        limit=None,
    ):
        self.toks_in_batch = toks_in_batch
        assert split.lower() in {
            "train",
            "val",
            "test",
            "tiny_train",
            "tiny_val",
            "tiny_test",
        }, "'split' must be in ['train', 'val', 'test', 'tiny_train', 'tiny_val', 'tiny_test'] !"
        self.split = split.lower()
        self.s_suffix = s_suffix
        self.t_suffix = t_suffix
        self.for_training = self.split.endswith("train")  # training ?

        self.tokenizer = tokenizer if tokenizer is not None else build_tokenizer()
        bos, eos = self.tokenizer.bos_token_id, self.tokenizer.eos_token_id

        file_path = os.path.join(data_dir, f"{split}.json")
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if limit is not None:
            data = data[:limit]  # e.g. a few pairs for an overfitting sanity check

        # tokenize once here (not at every batch of every epoch)
        self.data = list()
        self.n_dropped = 0
        for item in data:
            s, t = item[s_suffix].strip(), item[t_suffix].strip()
            if not s or not t:
                self.n_dropped += 1
                continue
            if self.t_suffix == "ar":
                t = t[::-1]  # right-to-left target: decode with is_ltr=True
            s_ids = self.tokenizer.encode(s)
            t_ids = [bos] + self.tokenizer.encode(t) + [eos]
            if max_seq_len is not None and (len(s_ids) > max_seq_len or len(t_ids) > max_seq_len):
                self.n_dropped += 1
                continue
            # [0, 1, 2, 3] <==> [src_ids, tgt_ids, src_lengths, tgt_lengths]
            self.data.append((s_ids, t_ids, len(s_ids), len(t_ids)))

        # Pre-sort by tgt lengths - required for itertools.groupby() later
        self.data.sort(key=lambda x: x[3])

        self.create_batches()

    def create_batches(self):
        chunks = [
            list(t) for _, t in groupby(self.data, key=lambda x: x[3])
        ]  # chunk by tgt sequence lengths
        self.batches = list()
        for chunk in chunks:
            chunk.sort(
                key=lambda x: x[2]
            )  # (samples with same src sequence length) ==> (in one batch)
            seqs_per_batch = max(
                1, self.toks_in_batch // chunk[0][3]
            )  # toks_in_batch // tgt_length ==> how many sentences fit in toks_in_batch
            self.batches.extend(
                [
                    chunk[i : i + seqs_per_batch]
                    for i in range(0, len(chunk), seqs_per_batch)
                ]
            )  # chunk ==> batches

        if self.for_training:
            shuffle(self.batches)  # Randomly shuffle batchs
        self.n_batches = len(self.batches)
        self.current_batch = -1

    def __len__(self):
        return self.n_batches

    def __iter__(self):
        self.current_batch = -1
        return self

    def __next__(self):
        self.current_batch += 1
        try:
            s_data, t_data, s_lens, t_lens = zip(*self.batches[self.current_batch])
        except IndexError:
            raise StopIteration

        s_data = pad_sequence(
            sequences=[torch.LongTensor(s) for s in s_data],
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )
        t_data = pad_sequence(
            sequences=[torch.LongTensor(t) for t in t_data],
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )

        s_lens = torch.LongTensor(s_lens)
        t_lens = torch.LongTensor(t_lens)

        return s_data, t_data, s_lens, t_lens

In [ ]:
%%writefile translation_engine/data/prepare_data.py
"""
Download a Darija <-> English parallel corpus and write the json files the DataLoader reads.

    python -m translation_engine.data.prepare_data

Source: the DODa sentences (Darija Open Dataset, Darija in Latin script) as packaged in
https://huggingface.co/datasets/atlasia/darija_english (config "doda", ~45k pairs).
Output (keys "dr" and "en", the default s_suffix / t_suffix):

    translation_engine/src/dataset/{train,val,test}.json
    translation_engine/src/dataset/{tiny_train,tiny_val,tiny_test}.json   (small copies for quick checks)

Any other corpus works too: write a json list of {"dr": "...", "en": "..."} objects.
"""

import argparse
import csv
import io
import json
import os
import random
import urllib.request

DODA_URL = "https://huggingface.co/datasets/atlasia/darija_english/resolve/main/atlasia_doda.csv"


def download_pairs(url: str = DODA_URL) -> list:
    """Return the list of {"dr", "en"} pairs from the DODa csv (empty rows and duplicates removed)."""
    with urllib.request.urlopen(url, timeout=120) as response:
        text = response.read().decode("utf-8")
    pairs, seen = [], set()
    for row in csv.DictReader(io.StringIO(text)):
        dr = " ".join((row.get("darija") or "").split())
        en = " ".join((row.get("en") or "").split())
        if dr and en and (dr, en) not in seen:
            seen.add((dr, en))
            pairs.append({"dr": dr, "en": en})
    return pairs


def split_pairs(pairs: list, val_ratio: float, test_ratio: float, seed: int) -> dict:
    """Shuffle once with a fixed seed, then cut into train / val / test."""
    pairs = list(pairs)
    random.Random(seed).shuffle(pairs)
    n_val = int(len(pairs) * val_ratio)
    n_test = int(len(pairs) * test_ratio)
    return {
        "val": pairs[:n_val],
        "test": pairs[n_val : n_val + n_test],
        "train": pairs[n_val + n_test :],
    }


def write_splits(splits: dict, out_dir: str, tiny_size: int) -> None:
    os.makedirs(out_dir, exist_ok=True)
    for name, items in splits.items():
        for prefix, subset in (("", items), ("tiny_", items[:tiny_size])):
            path = os.path.join(out_dir, f"{prefix}{name}.json")
            with open(path, "w", encoding="utf-8") as f:
                json.dump(subset, f, ensure_ascii=False, indent=0)
            print(f"{path}: {len(subset):,} pairs")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--out-dir", default="./translation_engine/src/dataset")
    parser.add_argument("--url", default=DODA_URL)
    parser.add_argument("--val-ratio", type=float, default=0.03)
    parser.add_argument("--test-ratio", type=float, default=0.03)
    parser.add_argument("--tiny-size", type=int, default=256)
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args(argv)

    pairs = download_pairs(args.url)
    print(f"downloaded {len(pairs):,} unique pairs")
    write_splits(split_pairs(pairs, args.val_ratio, args.test_ratio, args.seed), args.out_dir, args.tiny_size)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile translation_engine/model/__init__.py
"""BIALinguaNet: encoder, decoder, attention, loss and training utilities."""

In [ ]:
%%writefile translation_engine/model/ResFFNet.py
import torch
from torch import nn


class ResFFNet(nn.Module):
    """
    Residual Feedforward Network (ResFFNet): This module serves as a reasoning block, using a residual connection strategy
    to stabilize and enhance learning, particularly in deep neural networks.

    Attributes:
        n_emb (int): The embedding dimension of the input features.
        exp_fac (int): The expansion factor for the hidden layer, typically a multiple of the embedding dimension.
        d_rate (float): The dropout rate used to prevent overfitting.
    """

    def __init__(self, n_emb: int, exp_fac: int = 4, d_rate: float = 0.0) -> None:
        super(ResFFNet, self).__init__()
        self.n_emb = n_emb
        self.exp_fac = exp_fac
        self.d_rate = d_rate

        self.ln = nn.LayerNorm(self.n_emb)
        self.fc1 = nn.Linear(self.n_emb, exp_fac * self.n_emb)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(exp_fac * self.n_emb, self.n_emb)
        self.drop = nn.Dropout(d_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the ResFFNet module.

        Args:
            x (torch.Tensor): The input tensor of shape (B, T, n_emb), where B is the batch size, T is the sequence length,
                            and n_emb is the embedding dimension.

        Returns:
            torch.Tensor: The output tensor after applying the feedforward network and residual connection,
                            maintaining the input shape (B, T, n_emb).
        """

        o_x = x  # fork (no copy needed: x is not modified in place)
        x = self.ln(x)  # norm

        # feedforward
        x = self.drop(self.relu(self.fc1(x)))
        x = self.fc2(x)

        # dropout and res conn
        x = self.drop(x) + o_x
        return x

In [ ]:
%%writefile translation_engine/model/ResMHAtten.py
import torch
import torch.nn as nn
import math
from typing import Optional


class ResMHAtten(nn.Module):
    """
    A Multi-Head Attention mechanism with a residual connection, inspired by Convolutional Neural Networks (CNNs) used in vision tasks.

    This module can function as either self-attention or cross-attention, making it suitable for use in both the encoder
    and decoder parts of transformer models.

    Attributes:
        n_emb (int): The embedding dimension of the input features.
        n_head (int): The number of attention heads.
        h_size (int): The dimension of each attention head.
        is_decoder (bool): Indicates if the layer is used in a decoder, where masking of future tokens might be needed.
        device (str): Kept for backward compatibility; masks are built on the device of the inputs.
        d_rate (float): The dropout rate.
    """

    def __init__(
        self,
        n_emb: int,
        n_head: int,
        h_size: int,
        d_rate: float,
        is_decoder: bool = False,
        device: str = "cpu",
    ) -> None:
        super(ResMHAtten, self).__init__()
        self.n_emb = n_emb
        self.n_head = n_head
        self.h_size = h_size
        self.is_decoder = is_decoder
        self.device = device if torch.cuda.is_available() else 'cpu'

        self.q_proj = nn.Linear(n_emb, n_head * h_size)
        self.kv_proj = nn.Linear(n_emb, n_head * (h_size + h_size))
        self.o_proj = nn.Linear(n_head * h_size, n_emb)

        self.smax = nn.Softmax(dim=-1)
        self.ln = nn.LayerNorm(n_emb)
        self.drop = nn.Dropout(d_rate)

    def forward(
        self,
        q: torch.Tensor,
        kv: torch.Tensor,
        seq_len: torch.Tensor,
        is_self: Optional[bool] = None,
    ) -> torch.Tensor:
        """
        Forward pass of the ResMHAtten module.

        Args:
            q (torch.Tensor): The query tensor of shape (B, qT, n_emb), where B is the batch size and qT is the sequence length.
            kv (torch.Tensor): The key-value tensor of shape (B, kvT, n_emb), where kvT is the sequence length of the key-value pairs.
            seq_len (torch.Tensor): The actual length of each key-value sequence in the batch, shape (B,).
            is_self (bool, optional): True for self-attention, False for cross-attention.
                                      If None, it is self-attention when `kv` is the same tensor object as `q`.

        Returns:
            torch.Tensor: The output tensor after applying multi-head attention and residual connection, of shape (B, qT, n_emb).
        """

        B, qT, _ = q.shape
        _, kvT, _ = kv.shape

        # is self atten or cross atten (an explicit flag, not a value comparison:
        # comparing values costs a full pass over the tensors and can be fooled)
        if is_self is None:
            is_self = kv is q

        # fork for res con
        oq = q

        # pre-norm (diff than original tranfsormer)
        q = self.ln(q)
        kv = q if is_self else kv  # cross-attn: kv is the encoder output, already normalized

        # q, k, v --> qp, kp, vp    [B, xT, C]
        qp = self.q_proj(q)
        kp, vp = self.kv_proj(kv).split(split_size=self.n_head * self.h_size, dim=-1)

        # qp, kp, vp ==> (B, xT, nh, h)
        qp = qp.contiguous().view(B, qT, self.n_head, self.h_size)
        kp = kp.contiguous().view(B, kvT, self.n_head, self.h_size)
        vp = vp.contiguous().view(B, kvT, self.n_head, self.h_size)

        # [B, xT, nh, h] ==> [B * nh, xT, h]
        qp = qp.permute(0, 2, 1, 3).contiguous().view(-1, qT, self.h_size)
        kp = kp.permute(0, 2, 1, 3).contiguous().view(-1, kvT, self.h_size)
        vp = vp.permute(0, 2, 1, 3).contiguous().view(-1, kvT, self.h_size)

        # [B * nh, qT, h]   x   [B * nh, h, kvT]   ==>   [B * nh, qT, kvT]
        attn = torch.bmm(qp, kp.permute(0, 2, 1))
        attn = (1.0 / math.sqrt(self.h_size)) * attn  # /sqrt(h)

        # pad mask: key j is visible only if j < seq_len  [B * nh, 1, kvT]
        valid_pos = torch.arange(kvT, device=attn.device).view(1, 1, kvT)
        mask = valid_pos < seq_len.to(attn.device).repeat_interleave(self.n_head).view(-1, 1, 1)

        # self atten mask: query i only sees keys j <= i  [1, qT, kvT]
        if self.is_decoder and is_self:
            causal = torch.ones(qT, kvT, dtype=torch.bool, device=attn.device).tril()
            mask = mask & causal.unsqueeze(0)

        attn = attn.masked_fill(~mask, -float("inf"))

        # softmax + dropout
        attn = self.drop(self.smax(attn))

        # [B * nh, qT, kvT]   x   [B * nh, kvT, h]    ==>    [B * nh, qT, h]
        attn = torch.bmm(attn, vp)  # [B * nh, qT, h]
        attn = (
            attn.contiguous().view(B, self.n_head, qT, self.h_size).permute(0, 2, 1, 3)
        )  # [B, qT, nh, h]
        attn = attn.contiguous().view(B, qT, -1)  # [B, qT, nh * h]
        attn = self.o_proj(attn)  # [B, qT, e]

        # dropout + res-conn
        out = self.drop(attn) + oq
        return out

In [ ]:
%%writefile translation_engine/model/Encoder.py
import math
import torch
from torch import nn
from translation_engine.model.ResFFNet import ResFFNet
from translation_engine.model.ResMHAtten import ResMHAtten


class Encoder(nn.Module):
    """
    Encoder module for a transformer model, combining token embeddings, positional embeddings, and a series of
    multi-head attention and feedforward blocks with residual connections.

    This encoder is suitable for various sequence modeling tasks, providing a flexible architecture that can be
    customized with different numbers of layers, attention heads, and feedforward expansion factors.

    Attributes:
        ev_size (int): The size of the vocabulary or embedding vector.
        n_emb (int): The dimension of the embedding vector.
        n_head (int): The number of attention heads in the multi-head attention mechanism.
        h_size (int): The hidden size of each attention head.
        n_block (int): The number of blocks, each containing a multi-head attention and feedforward sub-layer.
        exp_fac (int): The expansion factor for the feedforward network.
        d_rate (float): The dropout rate used for regularization.
        device (str): The device on which the computations will be performed ('cpu' or 'cuda').
        max_seq_len (int): The maximum sequence length that the encoder can process.
    """

    def __init__(
        self,
        ev_size: int,
        n_emb: int,
        n_head: int,
        h_size: int,
        max_seq_len: int = 10000,
        n_block: int = 1,
        exp_fac: int = 4,
        d_rate: float = 0.0,
        device: str = "cpu",
    ):
        super(Encoder, self).__init__()

        self.ev_size = ev_size
        self.n_emb = n_emb
        self.n_head = n_head
        self.h_size = h_size
        self.n_block = n_block
        self.d_rate = d_rate
        self.exp_fac = exp_fac
        self.device = device if torch.cuda.is_available() else 'cpu'
        self.max_seq_len = max_seq_len

        self.tok_emb = nn.Embedding(ev_size, n_emb)
        self.pos_emb = nn.Embedding(max_seq_len, n_emb)

        self.blocks = nn.ModuleList(
            [
                nn.ModuleList(
                    [
                        ResMHAtten(
                            n_emb=n_emb,
                            n_head=n_head,
                            h_size=h_size,
                            d_rate=d_rate,
                            is_decoder=False,
                            device=device,
                        ),
                        ResFFNet(n_emb=n_emb, exp_fac=exp_fac, d_rate=d_rate),
                    ]
                )
                for _ in range(n_block)
            ]
        )

        self.drop = nn.Dropout(d_rate)
        self.ln = nn.LayerNorm(n_emb)

    def forward(self, idx: torch.Tensor, seq_len: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the Encoder module.

        Args:
            idx (torch.Tensor): Input tensor of token indices, shape (B, T), where B is the batch size and T is the sequence length.
            seq_len (torch.Tensor): Tensor containing the lengths of each sequence in the batch, used for masking.

        Returns:
            torch.Tensor: The encoded output tensor of shape (B, T, n_emb).

        Raises:
            ValueError: If the input sequence length T exceeds the maximum allowed sequence length `max_seq_len`.
        """

        B, T = idx.size()

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length T = {T} exceeds maximum allowed seq_len = {self.max_seq_len}"
            )

        pos = torch.arange(T, dtype=torch.long, device=idx.device)
        tok_emb = self.tok_emb(idx) * math.sqrt(self.n_emb)
        pos_emb = self.pos_emb(pos)

        x = self.drop(tok_emb + pos_emb)

        for attn, ffnet in self.blocks:
            x = attn(q=x, kv=x, seq_len=seq_len, is_self=True)
            x = ffnet(x)

        x = self.ln(x)
        return x

In [ ]:
%%writefile translation_engine/model/Decoder.py
import math
import torch
from torch import nn

from translation_engine.model.ResFFNet import ResFFNet
from translation_engine.model.ResMHAtten import ResMHAtten


class Decoder(nn.Module):
    """
    Decoder module for a transformer model, which combines token embeddings, positional embeddings, and a series of
    self-attention, cross-attention, and feedforward blocks with residual connections.

    This decoder is suitable for various sequence generation tasks, providing a flexible architecture that can be
    customized with different numbers of layers, attention heads, and feedforward expansion factors.

    Attributes:
        dv_size (int): The size of the target vocabulary.
        n_emb (int): The dimension of the embedding vector.
        n_head (int): The number of attention heads in the multi-head attention mechanism.
        h_size (int): The hidden size of each attention head.
        n_block (int): The number of blocks, each containing self-attention, cross-attention, and feedforward sub-layers.
        exp_fac (int): The expansion factor for the feedforward network.
        d_rate (float): The dropout rate used for regularization.
        device (str): The device on which the computations will be performed ('cpu' or 'cuda').
        max_seq_len (int): The maximum sequence length that the decoder can process.
    """

    def __init__(
        self,
        dv_size: int,
        n_emb: int,
        n_head: int,
        h_size: int,
        max_seq_len: int = 10000,
        n_block: int = 1,
        exp_fac: int = 4,
        d_rate: float = 0.0,
        device: str = "cpu",
    ):
        super(Decoder, self).__init__()

        self.dv_size = dv_size
        self.n_emb = n_emb
        self.n_head = n_head
        self.h_size = h_size
        self.n_block = n_block
        self.d_rate = d_rate
        self.exp_fac = exp_fac
        self.device = device if torch.cuda.is_available() else 'cpu'
        self.max_seq_len = max_seq_len

        self.tok_emb = nn.Embedding(dv_size, n_emb)
        self.pos_emb = nn.Embedding(max_seq_len, n_emb)

        self.blocks = nn.ModuleList(
            [
                nn.ModuleList(
                    [
                        ResMHAtten(
                            n_emb=n_emb,
                            n_head=n_head,
                            h_size=h_size,
                            d_rate=d_rate,
                            is_decoder=True,
                            device=device,
                        ),
                        ResMHAtten(
                            n_emb=n_emb,
                            n_head=n_head,
                            h_size=h_size,
                            d_rate=d_rate,
                            is_decoder=True,
                            device=device,
                        ),
                        ResFFNet(n_emb=n_emb, exp_fac=exp_fac, d_rate=d_rate),
                    ]
                )
                for _ in range(n_block)
            ]
        )

        self.drop = nn.Dropout(d_rate)
        self.ln = nn.LayerNorm(n_emb)
        self.fc = nn.Linear(n_emb, dv_size)

    def forward(
        self,
        d_idx: torch.Tensor,
        d_seq_len: torch.Tensor,
        e_x: torch.Tensor,
        e_seq_len: torch.Tensor,
    ) -> torch.Tensor:
        """
        Forward pass of the Decoder module.

        Args:
            d_idx (torch.Tensor): Input tensor of target token indices, shape (B, T), where B is the batch size and T is the sequence length.
            d_seq_len (torch.Tensor): Tensor containing the lengths of each sequence in the batch for the decoder, used for masking.
            e_x (torch.Tensor): Encoded input tensor from the encoder, shape (B, T_enc, n_emb).
            e_seq_len (torch.Tensor): Tensor containing the lengths of each sequence in the batch for the encoder, used for masking.

        Returns:
            torch.Tensor: The output tensor of shape (B, T, dv_size) representing the logits over the vocabulary for each time step.

        Raises:
            ValueError: If the input sequence length T exceeds the maximum allowed sequence length `max_seq_len`.
        """

        _, dT = d_idx.shape

        if dT > self.max_seq_len:
            raise ValueError(
                f"Sequence length dT = {dT} exceeds maximum allowed seq_len = {self.max_seq_len}"
            )

        pos = torch.arange(dT, dtype=torch.long, device=d_idx.device)
        tok_emb = self.tok_emb(d_idx) * math.sqrt(self.n_emb)
        pos_emb = self.pos_emb(pos)

        d_idx = self.drop(tok_emb + pos_emb)

        for self_attn, cross_attn, ffnet in self.blocks:
            d_idx = self_attn(q=d_idx, kv=d_idx, seq_len=d_seq_len, is_self=True)
            d_idx = cross_attn(q=d_idx, kv=e_x, seq_len=e_seq_len, is_self=False)
            d_idx = ffnet(d_idx)

        d_idx = self.ln(d_idx)
        d_idx = self.fc(d_idx)
        return d_idx

In [ ]:
%%writefile translation_engine/model/BIALinguaNet.py
import torch
from torch import nn
import torch.nn.functional as F
import math
from translation_engine.model.Decoder import Decoder
from translation_engine.model.Encoder import Encoder


class BIALinguaNet(nn.Module):
    """
    BIALinguaNet: A transformer-based architecture for bilingual language translation.

    This model consists of an encoder-decoder architecture, where the encoder processes the source language input,
    and the decoder generates the target language output. It supports various configurations for embedding size,
    number of attention heads, number of transformer blocks, and more.

    Attributes:
        ev_size (int): The size of the source language vocabulary.
        dv_size (int): The size of the target language vocabulary.
        n_emb (int): The dimension of the embedding vector.
        n_head (int): The number of attention heads in the multi-head attention mechanism.
        h_size (int): The hidden size of each attention head.
        n_block (int): The number of transformer blocks in both encoder and decoder.
        exp_fac (int): The expansion factor for the feedforward network.
        max_seq_len (int): The maximum sequence length that the model can process.
        d_rate (float): The dropout rate used for regularization.
        device (str): The device on which the computations will be performed ('cpu' or 'cuda').
    """

    def __init__(
        self,
        ev_size: int = 10000,
        dv_size: int = 10000,
        n_emb: int = 512,
        n_head: int = 8,
        h_size: int = 64,
        n_block: int = 6,
        exp_fac: int = 4,
        max_seq_len: int = 10000,
        d_rate: float = 0.0,
        device: str = "cpu",
    ):
        super(BIALinguaNet, self).__init__()

        self.ev_size = ev_size
        self.dv_size = dv_size
        self.n_emb = n_emb
        self.n_head = n_head
        self.h_size = h_size
        self.n_block = n_block
        self.exp_fac = exp_fac
        self.max_seq_len = max_seq_len
        self.d_rate = d_rate
        self.device = device if torch.cuda.is_available() else "cpu"

        self.encoder = Encoder(
            ev_size=ev_size,
            n_emb=n_emb,
            n_head=n_head,
            h_size=h_size,
            max_seq_len=max_seq_len,
            n_block=n_block,
            exp_fac=exp_fac,
            d_rate=d_rate,
            device=device,
        )

        self.decoder = Decoder(
            dv_size=dv_size,
            n_emb=n_emb,
            n_head=n_head,
            h_size=h_size,
            max_seq_len=max_seq_len,
            n_block=n_block,
            exp_fac=exp_fac,
            d_rate=d_rate,
            device=device,
        )

        self.init_weights()

    def init_weights(self) -> None:
        """
        Initialize the weights of the model using Xavier uniform initialization for linear layers and
        normal initialization for embeddings. The decoder's embedding and output layers are tied to
        share weights with the encoder's embedding layer.
        """

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p, gain=1.0)

        nn.init.normal_(
            self.encoder.tok_emb.weight, mean=0.0, std=math.pow(self.n_emb, -0.5)
        )
        self.decoder.tok_emb.weight = self.encoder.tok_emb.weight
        self.decoder.fc.weight = self.decoder.tok_emb.weight

    def forward(
        self,
        e_idx: torch.Tensor,
        d_idx: torch.Tensor,
        e_seq_len: torch.Tensor,
        d_seq_len: torch.Tensor,
    ) -> torch.Tensor:
        """
        Forward pass of the BIALinguaNet model.

        Args:
            e_idx (torch.Tensor): Input tensor of source language token indices, shape (B, T), where B is the batch size and T is the sequence length.
            d_idx (torch.Tensor): Input tensor of target language token indices for teacher forcing, shape (B, T).
            e_seq_len (torch.Tensor): Tensor containing the lengths of each sequence in the source batch.
            d_seq_len (torch.Tensor): Tensor containing the lengths of each sequence in the target batch.

        Returns:
            torch.Tensor: The output tensor of shape (B, T, dv_size) representing the logits over the target vocabulary.
        """

        e_idx = self.encoder(e_idx, e_seq_len)
        d_idx = self.decoder(d_idx, d_seq_len, e_idx, e_seq_len)
        return d_idx

    def get_init_args(self):
        """
        Get the initial arguments used to configure the model.

        Returns:
            dict: A dictionary containing the model's initialization parameters.
        """

        return {
            "ev_size": self.ev_size,
            "dv_size": self.dv_size,
            "n_emb": self.n_emb,
            "n_head": self.n_head,
            "h_size": self.h_size,
            "n_block": self.n_block,
            "exp_fac": self.exp_fac,
            "max_seq_len": self.max_seq_len,
            "d_rate": self.d_rate,
            "device": str(self.device),
        }

    @torch.no_grad()
    def translate(
        self,
        sx,
        tokenizer,
        temperature: float = 1.0,
        beam_size: int = 4,
        len_norm_coeff: float = 0.6,
        top_k: int = 50,
        top_p: float = 0.95,
        is_ltr: bool = False,
        max_beam_fork: int = 128,
    ):
        """
        Perform translation using beam search decoding.

        Args:
            sx (str or torch.Tensor): Source sentence, or a (1, T) tensor of source token ids.
            tokenizer: Tokenizer object for encoding/decoding sentences (with <PAD>, <SOS>, <EOS> added).
            temperature (float): Softmax temperature. 1.0 keeps the model's probabilities, < 1 sharpens, > 1 flattens.
            beam_size (int): Number of beams to use in beam search.
            len_norm_coeff (float): Length normalization coefficient (0 = no normalization, longer sentences are penalized).
            top_k (int): Each beam only expands its `top_k` most likely next tokens (0 = no limit).
            top_p (float): Each beam only expands the smallest set of tokens whose probability sums to `top_p` (1.0 = no limit).
            is_ltr (bool): Reverse each decoded string (for models trained on reversed targets, see the data loader).
            max_beam_fork (int): Maximum number of tokens to generate (also capped by the model's `max_seq_len`).

        Returns:
            tuple: The best hypothesis (translated sentence) and the list of all hypotheses with their scores.
        """

        device = next(self.parameters()).device
        self.eval()

        k = beam_size  # Beam size
        n_hypo = min(k, 10)  # n of hypotheses
        vs = self.dv_size  # Vocab size

        if isinstance(sx, str):
            ex = tokenizer.encode(sx)
            ex = torch.LongTensor(ex).unsqueeze(0)
        else:
            ex = sx
        ex = ex[:, : self.max_seq_len].to(device)  # longer inputs would overflow the position table
        if ex.size(1) == 0:
            return "", []
        e_seq_len = torch.LongTensor([ex.size(1)]).to(device)

        # Encoder Forward
        ex = self.encoder(idx=ex, seq_len=e_seq_len)

        hypo = torch.LongTensor([[tokenizer.bos_token_id]]).to(
            device
        )  # d_idx: <SOS>
        hypo_len = torch.LongTensor([hypo.size(1)]).to(device)  # d_seq_len: 1
        hypo_scores = torch.zeros(1).to(device)  # 1 score

        com_hypo = list()
        com_hypo_scores = list()

        # the decoder input has `step` tokens, so it must stay <= max_seq_len
        max_steps = max(1, min(max_beam_fork, self.max_seq_len))
        temperature = max(temperature, 1e-5)

        step = 1
        while True:
            s = hypo.size(0)  # s
            logits = self.decoder(
                d_idx=hypo,
                d_seq_len=hypo_len,
                e_x=ex.expand(s, -1, -1),
                e_seq_len=e_seq_len.expand(s),
            )  # [s, step, vs]
            flogits = self.top_k_top_p_filtering(
                logits=logits[:, -1, :], top_k=top_k, top_p=top_p
            )
            scores = flogits / temperature  # [s, vs]
            scores = F.log_softmax(scores, dim=-1)      # [s, vs]
            scores = hypo_scores.unsqueeze(1) + scores  # prev scores + curr scores

            # only expand candidates that survived top-k / top-p (finite score);
            # otherwise -inf "beams" with random tokens would fill the beam
            scores = scores.view(-1)
            n_cand = min(k, int(torch.isfinite(scores).sum().item()))
            top_k_hypo_scores, fttn_idx = scores.topk(n_cand, 0, True, True)  # [k]

            prev_tok_idx = fttn_idx // vs  # prev [k]
            next_tok_idx = fttn_idx % vs   # next [k]

            top_k_hypo = torch.cat(
                [hypo[prev_tok_idx], next_tok_idx.unsqueeze(1)], dim=1
            )  # [k, step + 1]

            complete = next_tok_idx == tokenizer.eos_token_id  # <EOS>? : [k], bool

            com_hypo.extend(top_k_hypo[complete].tolist())
            norm = math.pow(
                ((5 + step) / (5 + 1)), len_norm_coeff
            )  # chance(long sentence ~ short sentence)
            com_hypo_scores.extend((top_k_hypo_scores[complete] / norm).tolist())

            hypo = top_k_hypo[~complete]  # [s, step + 1] incomplete hypos
            hypo_scores = top_k_hypo_scores[~complete]  # d_idx(s): [s]

            # enough hypos, and no unfinished beam can still beat the best finished one
            # (stopping at the first n_hypo endings kept short, unlikely sentences
            # that happened to end early instead of the likely longer one)
            if len(com_hypo) >= n_hypo and (
                hypo.size(0) == 0 or hypo_scores.max().item() / norm <= max(com_hypo_scores)
            ):
                break
            hypo_len = torch.LongTensor(hypo.size(0) * [hypo.size(1)]).to(
                device
            )  # d_seq_len: [s]

            if hypo.size(0) == 0 or step >= max_steps:
                break  # every beam ended, or no EOS was found in time
            step += 1

        if len(com_hypo) == 0:  # in case no EOS is found
            com_hypo = hypo.tolist()
            com_hypo_scores = hypo_scores.tolist()

        all_hypos = list()  # idx ==> string
        spec_toks = [
            tokenizer.pad_token_id,
            tokenizer.bos_token_id,
            tokenizer.eos_token_id,
        ]
        if is_ltr:
            com_hypo = [
                tokenizer.decode([t for t in s if t not in spec_toks])[::-1]
                for s in com_hypo
            ]
        else:
            com_hypo = [
                tokenizer.decode([t for t in s if t not in spec_toks])
                for s in com_hypo
            ]

        for i, h in enumerate(com_hypo):
            all_hypos.append({"hypothesis": h, "score": com_hypo_scores[i]})

        max_idx = com_hypo_scores.index(max(com_hypo_scores))
        best_hypo = all_hypos[max_idx]["hypothesis"]

        return best_hypo, all_hypos

    def top_k_top_p_filtering(
        self,
        logits: torch.Tensor,
        top_k: int = 0,
        top_p: float = 1.0,
        filter_value: float = -float("Inf"),
        min_tokens_to_keep: int = 1,
    ) -> torch.Tensor:
        """
        Keep only the top-k tokens and/or the smallest set of tokens with cumulative probability >= top_p.

        Args:
            logits (torch.Tensor): Logits of shape (B, vocab_size). Not modified (a filtered copy is returned).
            top_k (int): Keep the `top_k` highest logits per row (0 = keep all).
            top_p (float): Nucleus threshold in (0, 1] (1.0 = keep all).
            filter_value (float): Value given to removed logits.
            min_tokens_to_keep (int): Always keep at least this many tokens per row.

        Returns:
            torch.Tensor: The filtered logits, same shape as `logits`.
        """
        logits = logits.clone()
        if top_k > 0:
            top_k = min(max(top_k, min_tokens_to_keep), logits.size(-1))
            indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
            logits[indices_to_remove] = filter_value

        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            if min_tokens_to_keep > 1:
                sorted_indices_to_remove[..., :min_tokens_to_keep] = 0
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[
                ..., :-1
            ].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices_to_remove.scatter(
                1, sorted_indices, sorted_indices_to_remove
            )
            logits[indices_to_remove] = filter_value
        return logits

In [ ]:
%%writefile translation_engine/model/LSRCrossEntropy.py
import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence


class LSRCrossEntropy(torch.nn.Module):
    """
    Label Smoothing Cross Entropy Loss (LSRCrossEntropy).

    This loss function applies label smoothing to the standard cross-entropy loss. Label smoothing can help to
    prevent the model from becoming too confident in its predictions, which can improve generalization.

    Attributes:
        eps (float): The smoothing parameter, which controls the amount of label smoothing applied.
        device (str): Kept for backward compatibility; the loss runs on the device of its inputs.
    """

    def __init__(self, eps: float = 0.1, device: str = "cpu"):
        super(LSRCrossEntropy, self).__init__()
        self.eps = eps
        self.device = device if torch.cuda.is_available() else 'cpu'

    def forward(self, x, y, lens):
        """
        Forward pass of the LSRCrossEntropy module.

        Args:
            x (torch.Tensor): The predicted logits tensor of shape (B, T, C), where B is the batch size,
                            T is the sequence length, and C is the number of classes.
            y (torch.Tensor): The target labels tensor of shape (B, T).
            lens (torch.Tensor): The lengths of each sequence in the batch, used to handle padding.

        Returns:
            torch.Tensor: The computed label smoothing cross-entropy loss.
        """

        lens = lens.cpu()  # pack_padded_sequence wants lengths on the CPU
        x = pack_padded_sequence(
            input=x, lengths=lens, batch_first=True, enforce_sorted=False
        ).data  # [n_tokens, C]: only the real (non-padding) positions
        y = pack_padded_sequence(
            input=y, lengths=lens, batch_first=True, enforce_sorted=False
        ).data  # [n_tokens]

        # smoothed target: (1 - eps) on the true class + eps spread over all C classes.
        # loss = -sum_c tv_c * log p_c = -(1 - eps) * log p_y - (eps / C) * sum_c log p_c,
        # computed without building the dense [n_tokens, C] target matrix (C = 50k here)
        log_p = F.log_softmax(x, dim=1)
        nll = -log_p.gather(dim=1, index=y.unsqueeze(1)).squeeze(1)
        smooth = -log_p.mean(dim=1)
        loss = (1.0 - self.eps) * nll + self.eps * smooth
        loss = torch.mean(loss)  # mean over real tokens

        return loss

In [ ]:
%%writefile translation_engine/model/utils.py
import os
import time
import torch
from IPython import get_ipython
from IPython.display import clear_output
import matplotlib.pyplot as plt
import math

from translation_engine.model.BIALinguaNet import BIALinguaNet


def get_lr(step: int, n_emb: int, warmup_steps: int) -> float:
    """
    "Noam" schedule from "Attention Is All You Need" (x2): linear warmup for
    `warmup_steps` optimizer steps, then decay proportional to 1/sqrt(step).
    """

    step = max(step, 1)  # step 0 would divide by zero
    sf = 2.0 * math.pow(n_emb, -0.5)  # Scale based on n_emb
    dp = math.pow(step, -0.5)  # LR during decay
    wp = step * math.pow(warmup_steps, -1.5)  # LR during warmup
    lr = sf * min(dp, wp)  # Final LR

    return lr


def save_checkpoint(
    epoch: int, model: torch.nn.Module, optimizer: torch.optim.Optimizer, file_path: str
):
    """Save model, optimizer and the model's init args (enough to rebuild it with load_checkpoint)."""
    folder = os.path.dirname(file_path)
    if folder:
        os.makedirs(folder, exist_ok=True)
    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "init_args": model.get_init_args(),
    }
    torch.save(state, file_path)


def load_checkpoint(
    file_path: str,
    device: torch.device,
    lr: float = 0.001,
    beta1: float = 0.9,
    beta2: float = 0.999,
    eps: float = 1e-8,
    weight_decay: float = 0.0,
):
    """Rebuild the model and its Adam optimizer from a checkpoint. Returns (model, optimizer, next_epoch)."""
    checkpoint = torch.load(file_path, map_location=device)
    init_args = checkpoint["init_args"]
    model = BIALinguaNet(**init_args)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer = torch.optim.Adam(
        params=[p for p in model.parameters() if p.requires_grad],
        lr=lr,
        betas=(beta1, beta2),
        eps=eps,
        weight_decay=weight_decay,
    )
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    model = model.to(device)
    return model, optimizer, start_epoch


def update_lr(optimizer: torch.optim.Optimizer, new_lr: float):
    for param_group in optimizer.param_groups:
        param_group["lr"] = new_lr


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.value = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, value: float, n: int = 1):
        self.value = value
        self.sum += value * n
        self.count += n
        self.avg = self.sum / self.count if self.count != 0 else 0


def in_notebook() -> bool:
    """True when running inside a Jupyter / Colab / Kaggle kernel."""
    shell = get_ipython()
    return shell is not None and "IPKernelApp" in getattr(shell, "config", {})


def live_plot_dual(data_dict, figsize=(12, 5), title="", save_path=None):
    """
    Plot train / validation loss per epoch. In a notebook the figure is redrawn in place;
    in a terminal it is saved to `save_path` instead (plt.show() would block training).
    """
    notebook = in_notebook()
    if notebook:
        clear_output(wait=True)
    fig, ax1 = plt.subplots(figsize=figsize)

    if data_dict["train_loss_avg"]:
        ax1.plot(data_dict["train_loss_avg"], "r-", label="Train Loss Avg")
        ax1.annotate(
            f"Avg: {data_dict['train_loss_avg'][-1]:.4f}",
            xy=(len(data_dict["train_loss_avg"]) - 1, data_dict["train_loss_avg"][-1]),
            xytext=(0, 10),
            textcoords="offset points",
            ha="center",
        )

    if data_dict["val_loss_avg"]:
        ax1.plot(data_dict["val_loss_avg"], "b-", label="Val Loss Avg")
        ax1.annotate(
            f"Avg: {data_dict['val_loss_avg'][-1]:.4f}",
            xy=(len(data_dict["val_loss_avg"]) - 1, data_dict["val_loss_avg"][-1]),
            xytext=(0, -15),
            textcoords="offset points",
            ha="center",
        )

    ax1.set_title("Training & Validation Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)

    plt.suptitle(title)
    if save_path:
        if os.path.dirname(save_path):
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=100)
    if notebook:
        plt.show()
    plt.close(fig)


def validator(val_loader, model, criterion, device):
    model.eval()

    with torch.no_grad():
        losses = AverageMeter()
        for sx, tx, sx_len, tx_len in val_loader:
            sx, tx, sx_len, tx_len = (
                sx.to(device),
                tx.to(device),
                sx_len.to(device),
                tx_len.to(device),
            )
            preds = model(sx, tx, sx_len, tx_len)
            loss = criterion(x=preds, y=tx[:, 1:], lens=tx_len - 1)
            losses.update(loss.item(), (tx_len - 1).sum().item())

        return losses


def trainer(
    train_loader,
    model,
    criterion,
    optimizer,
    epoch,
    step,
    device,
    batches_per_step,
    epochs,
    warmup_steps,
    n_emb,
    grad_clip=None,
    lr_scale=1.0,
    checkpoints_path=None,
):
    """
    Train for one epoch.

    Gradients are accumulated over `batches_per_step` batches, then the optimizer takes
    one step (so one step sees about `batches_per_step * toks_in_batch` target tokens).
    The learning rate follows get_lr(step) * lr_scale.

    Returns:
        (AverageMeter, int): the token-weighted training loss and the updated step counter.
    """
    model.train()
    data_time = AverageMeter()
    step_time = AverageMeter()
    losses = AverageMeter()

    optimizer.zero_grad()  # drop gradients left over from an incomplete accumulation
    start_data_time = time.time()
    start_step_time = time.time()

    for i, (sx, tx, sx_len, tx_len) in enumerate(train_loader):
        sx, tx, sx_len, tx_len = (
            sx.to(device),
            tx.to(device),
            sx_len.to(device),
            tx_len.to(device),
        )

        data_time.update(time.time() - start_data_time)

        preds = model(sx, tx, sx_len, tx_len)
        loss = criterion(x=preds, y=tx[:, 1:], lens=tx_len - 1)
        losses.update(loss.item(), (tx_len - 1).sum().item())

        # accumulate: each batch adds its share to the gradient of the step
        (loss / batches_per_step).backward()

        if (i + 1) % batches_per_step == 0:
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()
            optimizer.zero_grad()
            step += 1
            update_lr(
                optimizer,
                new_lr=get_lr(step=step, n_emb=n_emb, warmup_steps=warmup_steps) * lr_scale,
            )

            step_time.update(time.time() - start_step_time)

            start_step_time = time.time()

            # extra snapshots during the last 2 epochs (e.g. to average checkpoints)
            if checkpoints_path and (epoch in [epochs - 1, epochs - 2]) and (step % 1500 == 0):
                folder, name = os.path.split(checkpoints_path)
                save_checkpoint(
                    epoch, model, optimizer, file_path=os.path.join(folder, f"step{step}_{name}")
                )

        start_data_time = time.time()

    return losses, step


def build_model(config) -> BIALinguaNet:
    """A fresh BIALinguaNet with the sizes in `config`."""
    return BIALinguaNet(
        ev_size=config.ev_size,
        dv_size=config.dv_size,
        n_emb=config.n_emb,
        n_head=config.n_head,
        h_size=config.h_size,
        n_block=config.n_block,
        exp_fac=config.exp_fac,
        max_seq_len=config.max_seq_len,
        d_rate=config.d_rate,
        device=config.device,
    )


def fit(
    config,
    model,
    optimizer,
    criterion,
    train_loader,
    val_loader,
    start_epoch,
    epochs,
    lr_scale=1.0,
    save_path=None,
    plot_path=None,
    verbose=True,
    time_budget=None,
):
    """
    Train from `start_epoch` to `epochs`, validating after every epoch.
    The checkpoint at `save_path` is overwritten each time the validation loss improves,
    so it always holds the best model so far. Stops early after `time_budget` seconds
    (checked at the end of each epoch). Returns the loss history.
    """
    start_time = time.time()
    stats = {
        "train_loss_value": [],
        "train_loss_avg": [],
        "val_loss_value": [],
        "val_loss_avg": [],
    }
    best_val = math.inf
    steps_per_epoch = max(1, train_loader.n_batches // config.batches_per_step)

    for epoch in range(start_epoch, epochs):
        step = epoch * steps_per_epoch
        # set the lr for this point of the schedule (matters on resume / fine-tuning)
        update_lr(
            optimizer,
            get_lr(step=step, n_emb=config.n_emb, warmup_steps=config.warmup_steps) * lr_scale,
        )

        train_loader.create_batches()
        val_loader.create_batches()

        train_losses, step = trainer(
            train_loader=train_loader,
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            epoch=epoch,
            step=step,
            device=config.device,
            batches_per_step=config.batches_per_step,
            epochs=epochs,
            warmup_steps=config.warmup_steps,
            n_emb=config.n_emb,
            grad_clip=config.grad_clip,
            lr_scale=lr_scale,
            checkpoints_path=save_path,
        )

        val_losses = validator(
            val_loader=val_loader, model=model, criterion=criterion, device=config.device
        )

        stats["train_loss_value"].append(train_losses.value)
        stats["train_loss_avg"].append(train_losses.avg)
        stats["val_loss_value"].append(val_losses.value)
        stats["val_loss_avg"].append(val_losses.avg)

        live_plot_dual(stats, figsize=(12, 5), title=f"Epoch {epoch + 1}/{epochs}", save_path=plot_path)

        improved = val_losses.avg < best_val
        if improved:
            best_val = val_losses.avg
            if save_path:
                save_checkpoint(epoch, model, optimizer, file_path=save_path)
        if verbose:
            print(
                f"epoch {epoch + 1}/{epochs} | step {step} | lr {optimizer.param_groups[0]['lr']:.2e} | "
                f"train loss {train_losses.avg:.4f} | val loss {val_losses.avg:.4f}"
                + (" | saved" if improved and save_path else ""),
                flush=True,
            )
        if time_budget is not None and time.time() - start_time > time_budget:
            if verbose:
                print(f"time budget reached after epoch {epoch + 1}, stopping.")
            break

    return stats

## 3. Settings

- `DIRECTION`: `"dr-en"` (Darija → English) or `"en-dr"` (English → Darija).
- `SMOKE_TEST = True` runs a tiny version (256 pairs, a few epochs) to check the whole
  notebook in minutes, even on a CPU. Keep it `False` for real training.
- The model and optimisation settings are the repo's `Config` defaults; change them with
  `MODEL_OVERRIDES` (e.g. `dict(n_emb=256, n_head=8, n_block=4)` for a bigger model).

In [ ]:
import json
import math
import shutil
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch

sys.path.insert(0, os.getcwd())
from translation_engine.core.config import Config
from translation_engine.data import prepare_data
from translation_engine.data.dataloader import DataLoader, build_tokenizer
from translation_engine.model.LSRCrossEntropy import LSRCrossEntropy
from translation_engine.model.utils import build_model, fit, load_checkpoint

DIRECTION = "dr-en"         # "dr-en" or "en-dr"
SMOKE_TEST = False          # True = tiny quick check of the whole notebook
USE_GOOGLE_DRIVE = False    # Colab only: keep checkpoints on Drive (survive disconnects)
RESUME = True               # continue from the checkpoint in OUTPUT_DIR if it exists
TIME_BUDGET_HOURS = 11.0    # stop cleanly before the 12 h session limit
EPOCHS = None               # None = derived from Config.n_steps (100k optimizer steps)
OVERFIT_PAIRS = 32          # section 6: pairs the model must memorise
MODEL_OVERRIDES = dict()    # e.g. dict(n_emb=256, n_head=8, n_block=4)
SEED = 0

# --- where checkpoints go ------------------------------------------------------
ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ
if ON_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/bia-bilingual-net")
elif ON_KAGGLE:
    OUTPUT_DIR = Path("/kaggle/working/checkpoints")
else:
    OUTPUT_DIR = Path("checkpoints")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SRC, TGT = DIRECTION.split("-")
DATA_DIR = "translation_engine/src/dataset"
config = Config(
    s_suffix=SRC, t_suffix=TGT, data_dir=DATA_DIR, checkpoints_dir=str(OUTPUT_DIR),
    epochs=EPOCHS, **MODEL_OVERRIDES,
)
SPLITS = ("tiny_train", "tiny_val", "tiny_test") if SMOKE_TEST else ("train", "val", "test")
if SMOKE_TEST:
    config.epochs = 3
    OVERFIT_PAIRS, TIME_BUDGET_HOURS = 8, 0.5
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = config.cudnn_benchmark
print(config)

## 4. Data

[DODa](https://github.com/darija-open-dataset/dataset) sentence pairs (Darija written in Latin
script, e.g. *"bghit nmchi l dar"*), from
[atlasia/darija_english](https://huggingface.co/datasets/atlasia/darija_english).
`prepare_data` removes empty rows and duplicates, shuffles with a fixed seed and writes
`train / val / test` json files (94% / 3% / 3%).

The tokenizer is GPT-2's BPE plus `<PAD>`, `<SOS>`, `<EOS>`, shared by both languages (that is
what lets the model tie its input and output embeddings).

In [ ]:
if not Path(DATA_DIR, "train.json").exists():
    prepare_data.main(["--out-dir", DATA_DIR])

tokenizer = build_tokenizer(config.tokenizer_name)
config.set_vocab_size(tokenizer)
print("vocab size:", config.ev_size, "| pad / sos / eos ids:",
      tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id)

loader_args = dict(data_dir=DATA_DIR, s_suffix=SRC, t_suffix=TGT, toks_in_batch=config.toks_in_batch,
                   tokenizer=tokenizer, max_seq_len=config.max_seq_len)
train_loader = DataLoader(split=SPLITS[0], **loader_args)
val_loader = DataLoader(split=SPLITS[1], **loader_args)
print(f"train: {len(train_loader.data):,} pairs, {train_loader.n_batches:,} batches "
      f"({train_loader.n_dropped} dropped) | val: {len(val_loader.data):,} pairs")

### 4.1 Look at the data

In [ ]:
with open(Path(DATA_DIR, f"{SPLITS[0]}.json"), encoding="utf-8") as f:
    raw_pairs = json.load(f)
for pair in raw_pairs[:8]:
    print(f"{SRC}: {pair[SRC]:45}  {TGT}: {pair[TGT]}")

src_lens = [item[2] for item in train_loader.data]
tgt_lens = [item[3] for item in train_loader.data]
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist([src_lens, tgt_lens], bins=40, label=[f"source ({SRC})", f"target ({TGT}, with SOS/EOS)"])
ax.set_xlabel("tokens per sentence")
ax.legend()
plt.show()

sx, tx, s_len, t_len = next(iter(train_loader))
print("one batch:", tuple(sx.shape), tuple(tx.shape), "| target tokens:", int(t_len.sum()))
print("decoded target:", tokenizer.decode(tx[0, : t_len[0]].tolist()))

## 5. Model & neural-network sanity checks

Encoder-decoder Transformer with pre-norm residual blocks, learned positions and tied
embeddings. Before spending GPU hours we check, on one real batch, that:

1. the **initial loss ≈ ln(vocab)** (the untrained model guesses uniformly, it is not confidently wrong),
2. **every parameter gets a gradient**,
3. the decoder is **causal** (changing a future target token does not change earlier predictions),
4. **padding is ignored** (adding padding to the source does not change the output).

In [ ]:
device = config.device
model = build_model(config).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

criterion = LSRCrossEntropy(eps=0.0)
sx, tx, s_len, t_len = (t.to(device) for t in next(iter(val_loader)))

# 1. initial loss
model.eval()
with torch.no_grad():
    loss0 = criterion(model(sx, tx, s_len, t_len), tx[:, 1:], t_len - 1).item()
print(f"1. initial loss {loss0:.3f} vs ln(vocab) = {math.log(config.dv_size):.3f}")
assert abs(loss0 - math.log(config.dv_size)) < 1.5

# 2. gradients
model.train()
criterion(model(sx, tx, s_len, t_len), tx[:, 1:], t_len - 1).backward()
no_grad = [n for n, p in model.named_parameters() if p.grad is None or p.grad.abs().sum() == 0]
print("2. parameters without gradient:", no_grad or "none")
assert not no_grad
model.zero_grad()

# 3. causality and 4. padding, on one sentence
model.eval()
with torch.no_grad():
    s1, t1 = sx[:1, : s_len[0]], tx[:1, : t_len[0]]
    sl1, tl1 = s_len[:1], t_len[:1]
    base = model(s1, t1, sl1, tl1)
    changed = t1.clone()
    changed[0, -1] = (changed[0, -1] + 1) % config.dv_size
    assert torch.allclose(model(s1, changed, sl1, tl1)[:, :-1], base[:, :-1], atol=1e-4)
    print("3. causal decoder: ok")
    padded = torch.cat([s1, torch.full((1, 5), tokenizer.pad_token_id, device=device)], dim=1)
    assert torch.allclose(model(padded, t1, sl1, tl1), base, atol=1e-4)
    print("4. source padding ignored: ok")

## 6. Overfit a small chunk of data

The classic test before a long run: a fresh model, without dropout and label smoothing, must
**memorise** a few real pairs. The validation set here *is* those pairs, so the loss should drop
close to 0 and the beam search should give back the exact training translations. If it cannot
do this, something is broken and full training would be wasted time.

In [ ]:
overfit_config = Config(
    s_suffix=SRC, t_suffix=TGT, data_dir=DATA_DIR, d_rate=0.0, label_smoothing=0.0,
    batches_per_step=1, warmup_steps=400, **MODEL_OVERRIDES,
)
overfit_config.set_vocab_size(tokenizer)
overfit_loader = DataLoader(split=SPLITS[0], limit=OVERFIT_PAIRS, **loader_args)
overfit_model = build_model(overfit_config).to(device)
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=overfit_config.lr,
                                     betas=overfit_config.betas, eps=overfit_config.eps)
overfit_stats = fit(
    overfit_config, overfit_model, overfit_optimizer, LSRCrossEntropy(eps=0.0),
    overfit_loader, overfit_loader, start_epoch=0, epochs=20 if SMOKE_TEST else 60, verbose=False,
)
print(f"loss: {overfit_stats['val_loss_avg'][0]:.3f} -> {min(overfit_stats['val_loss_avg']):.4f}")

exact = 0
for pair in raw_pairs[:OVERFIT_PAIRS]:
    best, _ = overfit_model.translate(pair[SRC], tokenizer, beam_size=4)
    exact += best.strip() == pair[TGT].strip()
print(f"exact translations of the memorised pairs: {exact}/{OVERFIT_PAIRS}")
del overfit_model, overfit_optimizer

## 7. Training

Same loop as `python -m translation_engine.train`: label-smoothed cross-entropy, Adam with the
warmup + 1/√step ("Noam") learning-rate schedule, gradients accumulated over
`batches_per_step` batches (~25k target tokens per optimizer step), gradient clipping.
After each epoch it validates and saves the checkpoint when the validation loss improves.
The plot is redrawn after every epoch.

**Interrupted, or hit the session limit?** Run the notebook again with `RESUME = True`: it
continues from the saved checkpoint. On Kaggle, to continue in a *new* session, add the
previous version's output as input data and copy the checkpoint back to `/kaggle/working/checkpoints`.

In [ ]:
model = build_model(config)
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr, betas=config.betas,
                             eps=config.eps, weight_decay=config.weight_decay)
start_epoch = 0
if RESUME and Path(config.checkpoints_path).exists():
    model, optimizer, start_epoch = load_checkpoint(config.checkpoints_path, device=device)
    print(f"resumed from {config.checkpoints_path} at epoch {start_epoch}")
model = model.to(device)

steps_per_epoch = max(1, train_loader.n_batches // config.batches_per_step)
epochs = config.epochs or config.n_steps // steps_per_epoch + 1
print(f"{steps_per_epoch} optimizer steps per epoch, training until epoch {epochs}")

plot_path = OUTPUT_DIR / f"{SRC}_{TGT}_loss.png"
stats = fit(
    config, model, optimizer, LSRCrossEntropy(eps=config.label_smoothing, device=device),
    train_loader, val_loader, start_epoch=start_epoch, epochs=epochs,
    save_path=config.checkpoints_path, plot_path=str(plot_path),
    time_budget=TIME_BUDGET_HOURS * 3600,
)

## 8. Results

Loads the best checkpoint (exactly as the app does) and translates sentences it never trained
on: some from the test split, and a few everyday phrases.

In [ ]:
best_model, _, best_epoch = load_checkpoint(config.checkpoints_path, device=device)
print(f"best checkpoint: epoch {best_epoch}")


def translate(sentence, **kwargs):
    options = dict(beam_size=4, len_norm_coeff=0.6, top_k=50, top_p=0.95, temperature=1.0)
    options.update(kwargs)
    return best_model.translate(sentence, tokenizer, **options)


with open(Path(DATA_DIR, f"{SPLITS[2]}.json"), encoding="utf-8") as f:
    test_pairs = json.load(f)

print("--- test split (never seen in training) ---")
for pair in test_pairs[:15]:
    best, _ = translate(pair[SRC])
    print(f"{SRC}: {pair[SRC]}\n   model: {best}\n   human: {pair[TGT]}\n")

EXAMPLES = {
    "dr": ["fin awa ghadi?", "Mashi mochkil", "Chno katdir daba?", "bghit nmchi ldari",
           "smah lia, ma fhemtkch", "bghit nakol shi haja", "lah y3tik lkhir"],
    "en": ["where are you going?", "it's not a problem", "what are you doing now?",
           "I want to go home", "sorry, I don't understand", "I want to eat something", "thank you"],
}
print("--- everyday phrases ---")
for sentence in EXAMPLES[SRC]:
    best, hypos = translate(sentence)
    others = " | ".join(h["hypothesis"] for h in sorted(hypos, key=lambda h: -h["score"])[1:3])
    print(f"{sentence:30} -> {best:35} (other hypotheses: {others})")

## 9. Download the trained model

Zips the checkpoint and the loss curve. To use it with the repo's Streamlit app, put the
checkpoint in `translation_engine/src/checkpoints/` (the app expects `dr_en_chpts.pth.tar`
and `en_dr_chpts.pth.tar`), then run `streamlit run translation_engine/app/main.py`.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    for path in (Path(config.checkpoints_path), plot_path):
        if path.exists():
            shutil.copy(path, Path(tmp) / path.name)
    archive = shutil.make_archive(str(OUTPUT_DIR / f"bia-bilingual-net-{SRC}-{TGT}"), "zip", tmp)
print("created", archive)

if ON_COLAB:
    from google.colab import files

    files.download(archive)
elif ON_KAGGLE:
    print("Kaggle: click 'Save Version', then download it from the notebook's Output tab.")